In [ ]:
# setting root at top

import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

# OptimizedModel 224x224 Image and 5 Conv Blocks
# Using Class Weight + OverSampler Loss and 0.002 Learning Rate

### Import libraries
* `torch`
* `datasets` and `transforms` from `torchvision` for data Loading.
* `DataLoader` from `torch.utils.data` for Batching and using data in model.
* `OptimizedModel` from `architectures.OptimizedModel.py`
* `pandas` as `pd`
* `matplotlib.pyplot` as `plt`
* `seaborn` as sns
* `trainer` from `modules.TrainTest`
* `oversampler` from `modules.OverSampler`
* `compute_class_weights` from `modules.ClassWeights`

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from architectures.OptimizedModel import OptimizedModel

# visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#trainer and tester modules
from modules.TrainTest import trainer
from modules.ClassWeights import compute_class_weights
from modules.OverSampler import  oversampler

### Load datasets for training
* load 224x224 images
* load from `../dataset/final-dataset/train`
* convert to tensor
* make batch_size = 16

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.35),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3)
])

train_dataset = datasets.ImageFolder(
    root="../dataset/final-dataset/train",
    transform=train_transform
)

sampler = oversampler(dataset=train_dataset, oversample_rate=1)

In [ ]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    shuffle=False,
    sampler= sampler,
    pin_memory=True,
    batch_size=32,
    num_workers=2,
    persistent_workers=True    
    )

class_weight = compute_class_weights(train_dataloader)

### Load datasets for training
* load 224x224 images
* load from `../dataset/final-dataset/train`
* convert to tensor
* make batch_size = 32

In [ ]:
valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
valid_dataset = datasets.ImageFolder(
    root="../dataset/final-dataset/valid",
    transform=valid_transform
)
valid_dataloader = DataLoader(
    dataset=valid_dataset,
    batch_size=32,
    pin_memory=True,
    persistent_workers=True,
    num_workers = 2
)


## Use Model and Set Optimizers

* conv_layers = 4
* normalizations = [1,1,1,1]
* poolings = [1,1,1,1]
* dropouts = [0,1,0,1],
* initial_output_channel = 32,
* initial_image_size = 224,
* class_weights = class_weights,
* num_classes = 3

In [ ]:
model = OptimizedModel(
    conv_layers=4,
    normalizations = [1,1,1,1],
    poolings = [1,1,1,1],
    dropouts = [0,1,0,1],
    initial_output_channel = 32,
    initial_image_size = 224,
    class_weights = class_weight,
    num_classes = 3,
    dropout_p = 0.2
).to("cuda")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
model.set_optimizer(optimizer=optimizer)

### Use trainer to train model

In [ ]:
metrics = trainer(
    model = model,
    train_dataloader = train_dataloader,
    test_dataloader = valid_dataloader,
    epoch = 100,
    lr = 0.002,
    print_on=2,
    save_dir = "../models/opt_img224_c4_lr_0.002_b32",
    save_checkpoints=1,
    checkpoint_name = "opt_img224_c4_lr_0.002_b32_",
    early_stop_patience=20,
    monitor = "valid_f1",
    min_delta=0.0001
)

## Visualize

Convert metrics to a pd.DataFrame and add epoch column for visualization.

In [ ]:
metrics = pd.DataFrame(metrics)
epochs = [i+1 for i  in range(len(metrics))]
metrics["epoch"] = epochs

In [ ]:
metrics

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
axes = axes.flatten()

to_viz = [
    "loss",
    "accuracy",
    "precision",
    "recall",
    "f1"
]

for i, metric in enumerate(to_viz):

    ax = axes[i]

    sns.lineplot(
        data=metrics,
        x="epoch",
        y=f"training_{metric}",
        ax=ax,
        label="train"
    )

    sns.lineplot(
        data=metrics,
        x="epoch",
        y=f"valid_{metric}",
        ax=ax,
        label="valid"
    )

    ax.set_title(metric.capitalize())
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric.capitalize())

# Hide unused subplot (6th slot)
axes[-1].axis("off")

plt.tight_layout()
plt.show()